In [ ]:
import os
import random
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
import torchvision.transforms.functional as TF
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ===== Config =====
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

data_dir = '../../data/marker_gene_spatial_expression'
model_path = '../../model/marker_gene_spatial_expression/Xenium/'
os.makedirs(model_path, exist_ok=True)

NUM_GENES = 19
BATCH_SIZE = 16
NUM_EPOCHS = 1000
LR = 2e-4
NUM_WORKERS = 4
ENCODER_NAME = 'tu-convnext_tiny'

MARKER_GENES = [
    # Epithelial
    "EPCAM", "EGFR",
    # Fibroblast / Stromal
    "ACTA2", "PDGFRA", "PDGFRB", "SFRP4",
    # Endothelial
    "PECAM1",
    # Macrophage / Myeloid
    "CD68", "AIF1", "FCGR3A", "MRC1",
    # T cell
    "CD3E", "CD4", "CD8A", "TRAC",
    # B cell
    "CD79A", "MS4A1", "BANK1", "TCL1A"
]

In [ ]:
# ===== Augmentation =====
def apply_color_augmentation(image):
    """H&E 병리 이미지용 color augmentation (numpy uint8 HWC)"""
    # Brightness (50%)
    if random.random() < 0.5:
        f = random.uniform(0.8, 1.2)
        image = np.clip(image * f, 0, 255).astype(np.uint8)

    # Contrast (50%)
    if random.random() < 0.5:
        f = random.uniform(0.8, 1.2)
        mean = image.mean()
        image = np.clip((image - mean) * f + mean, 0, 255).astype(np.uint8)

    # Hue shift (50%)
    if random.random() < 0.5:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] = np.clip(hsv[:, :, 0] + random.uniform(-10, 10), 0, 179)
        image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    # Saturation (50%)
    if random.random() < 0.5:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.7, 1.3), 0, 255)
        image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    # Gamma correction (30%)
    if random.random() < 0.3:
        inv_gamma = 1.0 / random.uniform(0.8, 1.2)
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype(np.uint8)
        image = cv2.LUT(image, table)

    # Channel-wise brightness (30%)
    if random.random() < 0.3:
        for c in range(3):
            image[:, :, c] = np.clip(
                image[:, :, c].astype(np.float32) + random.uniform(-10, 10), 0, 255
            ).astype(np.uint8)

    # Gaussian noise (30%)
    if random.random() < 0.3:
        noise = np.random.normal(0, random.uniform(0, 5), image.shape)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # Gaussian blur (20%)
    if random.random() < 0.2:
        ks = random.choice([3, 5])
        image = cv2.GaussianBlur(image, (ks, ks), 0)

    return image


# ===== Dataset =====
class MultiScaleRegressionDataset(Dataset):
    def __init__(self, img_20x_list, img_5x_list, label_prop_list, label_int_list, augment=False):
        self.img_20x_paths = img_20x_list
        self.img_5x_paths = img_5x_list
        self.label_prop_paths = label_prop_list
        self.label_int_paths = label_int_list
        self.augment = augment
        self.to_tensor = ToTensor()

    def __len__(self):
        return len(self.img_20x_paths)

    def __getitem__(self, idx):
        img_20x = np.array(Image.open(self.img_20x_paths[idx]).convert('RGB'))
        img_5x = np.array(Image.open(self.img_5x_paths[idx]).convert('RGB'))
        label_prop = np.load(self.label_prop_paths[idx]).astype(np.float32)
        label_int = np.load(self.label_int_paths[idx]).astype(np.float32)

        # NaN → 0, clamp to [0, 1]
        label_prop = np.nan_to_num(label_prop, nan=0.0, posinf=1.0, neginf=0.0)
        label_int = np.nan_to_num(label_int, nan=0.0, posinf=1.0, neginf=0.0)
        label_prop = np.clip(label_prop, 0.0, 1.0)
        label_int = np.clip(label_int, 0.0, 1.0)

        label_prop = torch.from_numpy(label_prop)
        label_int = torch.from_numpy(label_int)

        if self.augment:
            # --- Geometric (동일하게 적용) ---
            # Horizontal flip (50%)
            if random.random() < 0.5:
                img_20x = np.fliplr(img_20x).copy()
                img_5x = np.fliplr(img_5x).copy()

            # Vertical flip (50%)
            if random.random() < 0.5:
                img_20x = np.flipud(img_20x).copy()
                img_5x = np.flipud(img_5x).copy()

            # 90° rotation (30%)
            if random.random() < 0.3:
                k = random.randint(1, 3)
                img_20x = np.rot90(img_20x, k).copy()
                img_5x = np.rot90(img_5x, k).copy()

            # --- Color (각각 독립 적용: 스케일별 staining 차이 반영) ---
            img_20x = apply_color_augmentation(img_20x)
            img_5x = apply_color_augmentation(img_5x)

        img_20x = self.to_tensor(img_20x.copy())
        img_5x = self.to_tensor(img_5x.copy())

        return img_20x, img_5x, label_prop, label_int


# ===== File list =====
img_20x_list = sorted(glob(f'{data_dir}/image/Xenium/**/*.tiff', recursive=True))
print(f'Total 20x patches: {len(img_20x_list)}')

img_5x_list = [p.replace('/image/', '/image_5x/') for p in img_20x_list]
label_prop_list = [p.replace('/image/', '/label_proportion/').replace('.tiff', '.npy') for p in img_20x_list]
label_int_list = [p.replace('/image/', '/label_intensity/').replace('.tiff', '.npy') for p in img_20x_list]

# 존재하지 않는 파일 필터링
valid_idx = [i for i in range(len(img_20x_list))
             if os.path.exists(img_5x_list[i])
             and os.path.exists(label_prop_list[i])
             and os.path.exists(label_int_list[i])]
print(f'Valid samples: {len(valid_idx)} / {len(img_20x_list)}')

img_20x_list = [img_20x_list[i] for i in valid_idx]
img_5x_list = [img_5x_list[i] for i in valid_idx]
label_prop_list = [label_prop_list[i] for i in valid_idx]
label_int_list = [label_int_list[i] for i in valid_idx]

# Train / Val split
indices = list(range(len(img_20x_list)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}')

train_dataset = MultiScaleRegressionDataset(
    [img_20x_list[i] for i in train_idx], [img_5x_list[i] for i in train_idx],
    [label_prop_list[i] for i in train_idx], [label_int_list[i] for i in train_idx],
    augment=True
)
val_dataset = MultiScaleRegressionDataset(
    [img_20x_list[i] for i in val_idx], [img_5x_list[i] for i in val_idx],
    [label_prop_list[i] for i in val_idx], [label_int_list[i] for i in val_idx],
    augment=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# ===== Model =====
class MultiScaleRegressionModel(nn.Module):
    def __init__(self, encoder_name='tu-convnext_tiny', num_genes=19):
        super().__init__()
        self.encoder_20x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')
        self.encoder_5x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')

        # ConvNeXt-Tiny last stage output channels
        enc_channels = self.encoder_20x.out_channels  # e.g. (3, 96, 192, 384, 768, 768)
        feat_dim = enc_channels[-1]  # 768

        # Head A: Proportion (logits — Sigmoid은 loss에서 처리)
        self.head_prop = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
        )

        # Head B: Intensity
        self.head_int = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
            nn.Sigmoid()
        )

    def forward(self, x_20x, x_5x):
        feat_20x = self.encoder_20x(x_20x)  # list of feature maps
        feat_5x = self.encoder_5x(x_5x)

        # Fuse last stage features
        fused = feat_20x[-1] + feat_5x[-1]  # (B, 768, H, W)

        # Global Average Pooling
        pooled = F.adaptive_avg_pool2d(fused, 1).flatten(1)  # (B, 768)

        prop_logits = self.head_prop(pooled)  # (B, 15) — raw logits
        intensity = self.head_int(pooled)      # (B, 15) — [0, 1]

        return prop_logits, intensity


# ===== Loss =====
def pearson_corr_loss(pred, target):
    """1 - mean PCC across genes. pred, target: (B, G)"""
    pred_c = pred - pred.mean(dim=0, keepdim=True)
    target_c = target - target.mean(dim=0, keepdim=True)
    cov = (pred_c * target_c).sum(dim=0)
    pred_std = pred_c.pow(2).sum(dim=0).sqrt()
    target_std = target_c.pow(2).sum(dim=0).sqrt()
    pcc = cov / (pred_std * target_std + 1e-8)
    return 1.0 - pcc.mean()  # 1 - mean PCC


class TwoHeadLoss(nn.Module):
    """
    Proportion: BCEWithLogitsLoss (수치적으로 안정적, logits 입력)
    Intensity:  MSE
    PCC Loss:   1 - Pearson correlation (직접 상관관계 최적화)

    total = α × BCE(prop) + β × MSE(int) + γ × PCC_loss(both heads)
    """
    def __init__(self, prop_weight=1.0, int_weight=1.0, pcc_weight=0.5):
        super().__init__()
        self.prop_weight = prop_weight
        self.int_weight = int_weight
        self.pcc_weight = pcc_weight
        self.bce = nn.BCEWithLogitsLoss()  # logits → 내부에서 sigmoid 적용
        self.mse = nn.MSELoss()

    def forward(self, pred_prop_logits, pred_int, gt_prop, gt_int):
        # Clamp targets for safety
        gt_prop = gt_prop.clamp(0.0, 1.0)
        gt_int = gt_int.clamp(0.0, 1.0)

        loss_prop = self.bce(pred_prop_logits, gt_prop)
        loss_int = self.mse(pred_int, gt_int)

        # PCC loss: proportion은 sigmoid 적용 후 계산
        pred_prop_sigmoid = torch.sigmoid(pred_prop_logits)
        pcc_prop = pearson_corr_loss(pred_prop_sigmoid, gt_prop)
        pcc_int = pearson_corr_loss(pred_int, gt_int)
        loss_pcc = (pcc_prop + pcc_int) / 2.0

        total = (self.prop_weight * loss_prop
                 + self.int_weight * loss_int
                 + self.pcc_weight * loss_pcc)
        return total, loss_prop, loss_int, loss_pcc


# ===== PCC metric =====
def pearson_corr_vector(pred, target):
    """Per-gene Pearson correlation. pred, target: (B, 15) -> returns (15,)"""
    pred_c = pred - pred.mean(dim=0, keepdim=True)
    target_c = target - target.mean(dim=0, keepdim=True)
    cov = (pred_c * target_c).sum(dim=0)
    pred_std = pred_c.pow(2).sum(dim=0).sqrt()
    target_std = target_c.pow(2).sum(dim=0).sqrt()
    pcc = cov / (pred_std * target_std + 1e-8)
    return pcc  # (15,)


model = MultiScaleRegressionModel(ENCODER_NAME, NUM_GENES).to(device)
criterion = TwoHeadLoss(prop_weight=1.0, int_weight=1.0, pcc_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')
print(f'Loss: BCEWithLogits(prop) + MSE(int) + 0.5 × PCC_loss')

# Load checkpoint (with NaN validation)
ckpt_path = os.path.join(model_path, 'train_2head_best.pt')
if os.path.exists(ckpt_path):
    state_dict = torch.load(ckpt_path, map_location=device)
    has_nan = any(torch.isnan(v).any().item() for v in state_dict.values() if v.is_floating_point())
    has_inf = any(torch.isinf(v).any().item() for v in state_dict.values() if v.is_floating_point())
    if has_nan or has_inf:
        print(f'[WARNING] Checkpoint contains {"NaN" if has_nan else ""}{"+" if has_nan and has_inf else ""}{"Inf" if has_inf else ""} weights! Training from scratch.')
    else:
        model.load_state_dict(state_dict, strict=False)
        print(f'Checkpoint loaded: {ckpt_path}')
else:
    print('No checkpoint found. Training from scratch.')

# Shape test
with torch.no_grad():
    dummy_20x = torch.randn(2, 3, 512, 512).to(device)
    dummy_5x = torch.randn(2, 3, 512, 512).to(device)
    p, i = model(dummy_20x, dummy_5x)
    print(f'Output shapes: prop_logits={p.shape}, int={i.shape}')
    print(f'prop range: [{p.min():.3f}, {p.max():.3f}], int range: [{i.min():.3f}, {i.max():.3f}]')

In [ ]:
# ===== Training Loop =====
train_loss_list, val_loss_list = [], []
train_prop_loss_list, val_prop_loss_list = [], []
train_int_loss_list, val_int_loss_list = [], []
train_pcc_loss_list, val_pcc_loss_list = [], []
train_pcc_prop_list, val_pcc_prop_list = [], []
train_pcc_int_list, val_pcc_int_list = [], []
MIN_loss = float('inf')
MAX_GRAD_NORM = 1.0  # gradient clipping

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    running_loss, running_prop_loss, running_int_loss, running_pcc_loss = 0, 0, 0, 0
    all_pred_prop, all_gt_prop = [], []
    all_pred_int, all_gt_int = [], []
    n_valid_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:4d} [Train]', leave=False)
    for img_20x, img_5x, gt_prop, gt_int in pbar:
        img_20x = img_20x.to(device)
        img_5x = img_5x.to(device)
        gt_prop = gt_prop.to(device)
        gt_int = gt_int.to(device)

        optimizer.zero_grad()
        pred_prop_logits, pred_int = model(img_20x, img_5x)
        loss, lp, li, lpcc = criterion(pred_prop_logits, pred_int, gt_prop, gt_int)

        # NaN loss → skip batch
        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  [!] NaN/Inf loss detected, skipping batch')
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

        n_valid_batches += 1
        running_loss += loss.item()
        running_prop_loss += lp.item()
        running_int_loss += li.item()
        running_pcc_loss += lpcc.item()
        # metric 계산용: logits → sigmoid
        all_pred_prop.append(torch.sigmoid(pred_prop_logits).detach().cpu())
        all_gt_prop.append(gt_prop.cpu())
        all_pred_int.append(pred_int.detach().cpu())
        all_gt_int.append(gt_int.cpu())

        pbar.set_postfix(loss=f'{running_loss / n_valid_batches:.4f}')

    n_train = max(n_valid_batches, 1)
    train_loss_list.append(running_loss / n_train)
    train_prop_loss_list.append(running_prop_loss / n_train)
    train_int_loss_list.append(running_int_loss / n_train)
    train_pcc_loss_list.append(running_pcc_loss / n_train)

    if all_pred_prop:
        all_pred_prop = torch.cat(all_pred_prop)
        all_gt_prop = torch.cat(all_gt_prop)
        all_pred_int = torch.cat(all_pred_int)
        all_gt_int = torch.cat(all_gt_int)
        train_pcc_prop_list.append(pearson_corr_vector(all_pred_prop, all_gt_prop).mean().item())
        train_pcc_int_list.append(pearson_corr_vector(all_pred_int, all_gt_int).mean().item())
    else:
        train_pcc_prop_list.append(0.0)
        train_pcc_int_list.append(0.0)

    # --- Validation ---
    model.eval()
    running_loss, running_prop_loss, running_int_loss, running_pcc_loss = 0, 0, 0, 0
    all_pred_prop, all_gt_prop = [], []
    all_pred_int, all_gt_int = [], []

    with torch.no_grad():
        for img_20x, img_5x, gt_prop, gt_int in tqdm(val_loader, desc=f'Epoch {epoch:4d} [Val]', leave=False):
            img_20x = img_20x.to(device)
            img_5x = img_5x.to(device)
            gt_prop = gt_prop.to(device)
            gt_int = gt_int.to(device)

            pred_prop_logits, pred_int = model(img_20x, img_5x)
            loss, lp, li, lpcc = criterion(pred_prop_logits, pred_int, gt_prop, gt_int)

            running_loss += loss.item()
            running_prop_loss += lp.item()
            running_int_loss += li.item()
            running_pcc_loss += lpcc.item()
            all_pred_prop.append(torch.sigmoid(pred_prop_logits).cpu())
            all_gt_prop.append(gt_prop.cpu())
            all_pred_int.append(pred_int.cpu())
            all_gt_int.append(gt_int.cpu())

    n_val = len(val_loader)
    val_total = running_loss / n_val
    val_loss_list.append(val_total)
    val_prop_loss_list.append(running_prop_loss / n_val)
    val_int_loss_list.append(running_int_loss / n_val)
    val_pcc_loss_list.append(running_pcc_loss / n_val)

    all_pred_prop = torch.cat(all_pred_prop)
    all_gt_prop = torch.cat(all_gt_prop)
    all_pred_int = torch.cat(all_pred_int)
    all_gt_int = torch.cat(all_gt_int)
    val_pcc_prop = pearson_corr_vector(all_pred_prop, all_gt_prop).mean().item()
    val_pcc_int = pearson_corr_vector(all_pred_int, all_gt_int).mean().item()
    val_pcc_prop_list.append(val_pcc_prop)
    val_pcc_int_list.append(val_pcc_int)
    print(f'Epoch {epoch:4d} | '
            f'Train: {train_loss_list[-1]:.4f} (BCE:{train_prop_loss_list[-1]:.4f} MSE:{train_int_loss_list[-1]:.4f} PCC:{train_pcc_loss_list[-1]:.4f}) | '
            f'Val: {val_loss_list[-1]:.4f} (BCE:{val_prop_loss_list[-1]:.4f} MSE:{val_int_loss_list[-1]:.4f} PCC:{val_pcc_loss_list[-1]:.4f}) | '
            f'PCC prop: {val_pcc_prop:.4f} int: {val_pcc_int:.4f}')    
    # --- Checkpoint ---
    if val_total < MIN_loss and not np.isnan(val_total):
        MIN_loss = val_total
        torch.save(model.state_dict(), f'{model_path}train_2head_best.pt')




    # --- Plot ---
    if epoch % 100 == 99:
        fig, axes = plt.subplots(1, 4, figsize=(24, 5))

        ax = axes[0]
        ax.plot(train_loss_list, label='Train Total')
        ax.plot(val_loss_list, label='Val Total')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Total Loss'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[1]
        ax.plot(train_prop_loss_list, label='Train BCE(prop)', alpha=0.7)
        ax.plot(val_prop_loss_list, label='Val BCE(prop)', alpha=0.7)
        ax.plot(train_int_loss_list, label='Train MSE(int)', alpha=0.7, linestyle='--')
        ax.plot(val_int_loss_list, label='Val MSE(int)', alpha=0.7, linestyle='--')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Per-Head Loss'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[2]
        ax.plot(train_pcc_loss_list, label='Train PCC Loss', alpha=0.7)
        ax.plot(val_pcc_loss_list, label='Val PCC Loss', alpha=0.7)
        ax.set_xlabel('Epoch'); ax.set_ylabel('1 - PCC')
        ax.set_title('PCC Loss (1 - corr)'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[3]
        ax.plot(train_pcc_prop_list, label='Train Prop PCC', alpha=0.7)
        ax.plot(val_pcc_prop_list, label='Val Prop PCC', alpha=0.7)
        ax.plot(train_pcc_int_list, label='Train Int PCC', alpha=0.7, linestyle='--')
        ax.plot(val_pcc_int_list, label='Val Int PCC', alpha=0.7, linestyle='--')
        ax.set_xlabel('Epoch'); ax.set_ylabel('PCC')
        ax.set_title('Pearson Correlation'); ax.legend(); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

print(f'\nBest val loss: {MIN_loss:.6f}')
print(f'Model saved: {model_path}train_2head_best.pt')

In [ ]:
# ===== Evaluation =====
model.load_state_dict(torch.load(f'{model_path}train_2head_best.pt', map_location=device))
model.eval()

all_pred_prop, all_gt_prop = [], []
all_pred_int, all_gt_int = [], []

with torch.no_grad():
    for img_20x, img_5x, gt_prop, gt_int in tqdm(val_loader, desc='Evaluation', leave=True):
        img_20x = img_20x.to(device)
        img_5x = img_5x.to(device)
        pred_prop_logits, pred_int = model(img_20x, img_5x)
        all_pred_prop.append(torch.sigmoid(pred_prop_logits).cpu())
        all_gt_prop.append(gt_prop)
        all_pred_int.append(pred_int.cpu())
        all_gt_int.append(gt_int)

all_pred_prop = torch.cat(all_pred_prop)
all_gt_prop = torch.cat(all_gt_prop)
all_pred_int = torch.cat(all_pred_int)
all_gt_int = torch.cat(all_gt_int)

pcc_prop = pearson_corr_vector(all_pred_prop, all_gt_prop)  # (19,)
pcc_int = pearson_corr_vector(all_pred_int, all_gt_int)  # (19,)
mae_prop = (all_pred_prop - all_gt_prop).abs().mean(dim=0)  # (19,)
mae_int = (all_pred_int - all_gt_int).abs().mean(dim=0)  # (19,)

print(f'\n{"Gene":>10s} | {"PCC(prop)":>10s} {"MAE(prop)":>10s} | {"PCC(int)":>10s} {"MAE(int)":>10s}')
print('-' * 60)
for gi, gene in enumerate(MARKER_GENES):
    print(f'{gene:>10s} | {pcc_prop[gi]:10.4f} {mae_prop[gi]:10.4f} | {pcc_int[gi]:10.4f} {mae_int[gi]:10.4f}')
print('-' * 60)
print(f'{"Mean":>10s} | {pcc_prop.mean():10.4f} {mae_prop.mean():10.4f} | {pcc_int.mean():10.4f} {mae_int.mean():10.4f}')

# ===== 1. Per-Gene PCC & MAE Bar Chart =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(NUM_GENES)
w = 0.35

ax = axes[0]
ax.bar(x - w/2, pcc_prop.numpy(), w, label='Proportion', color='steelblue')
ax.bar(x + w/2, pcc_int.numpy(), w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('PCC'); ax.set_title('Per-Gene Pearson Correlation')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(x - w/2, mae_prop.numpy(), w, label='Proportion', color='steelblue')
ax.bar(x + w/2, mae_int.numpy(), w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('MAE'); ax.set_title('Per-Gene Mean Absolute Error')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# ===== 2. Scatter Plot: Pred vs GT (per gene, Proportion & Intensity) =====
fig, axes = plt.subplots(3, 5, figsize=(25, 15))
fig.suptitle('Proportion: Predicted vs Ground Truth', fontsize=16, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_prop[:, gi].numpy()
    pred_vals = all_pred_prop[:, gi].numpy()
    ax.scatter(gt_vals, pred_vals, alpha=0.15, s=8, color='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, alpha=0.7)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    ax.set_title(f'{gene}\nPCC={pcc_prop[gi]:.3f}', fontsize=10)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 5, figsize=(25, 15))
fig.suptitle('Intensity: Predicted vs Ground Truth', fontsize=16, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_int[:, gi].numpy()
    pred_vals = all_pred_int[:, gi].numpy()
    ax.scatter(gt_vals, pred_vals, alpha=0.15, s=8, color='coral')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, alpha=0.7)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    ax.set_title(f'{gene}\nPCC={pcc_int[gi]:.3f}', fontsize=10)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ===== 3. Error Distribution Boxplot =====
error_prop = (all_pred_prop - all_gt_prop).numpy()  # (N, 19)
error_int = (all_pred_int - all_gt_int).numpy()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax = axes[0]
bp = ax.boxplot([error_prop[:, gi] for gi in range(NUM_GENES)],
                labels=MARKER_GENES, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)
ax.axhline(y=0, color='red', linewidth=0.8, linestyle='--')
ax.set_ylabel('Prediction Error (Pred - GT)')
ax.set_title('Proportion Error Distribution')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
bp = ax.boxplot([error_int[:, gi] for gi in range(NUM_GENES)],
                labels=MARKER_GENES, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('coral')
    patch.set_alpha(0.6)
ax.axhline(y=0, color='red', linewidth=0.8, linestyle='--')
ax.set_ylabel('Prediction Error (Pred - GT)')
ax.set_title('Intensity Error Distribution')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# ===== 4. Gene Group Summary =====
gene_groups = {
    'Epithelial/Tumor': [0, 1],
    'Stromal/Fibroblast': [2, 3, 4, 5],
    'Endothelial': [6],
    'Macrophage / Myeloid': [7, 8, 9, 10],
    'T Cell': [11, 12, 13, 14],
    'B Cell': [15, 16, 17, 18]
}

group_names = list(gene_groups.keys())
group_pcc_prop = [pcc_prop[idx].mean().item() for idx in gene_groups.values()]
group_pcc_int = [pcc_int[idx].mean().item() for idx in gene_groups.values()]
group_mae_prop = [mae_prop[idx].mean().item() for idx in gene_groups.values()]
group_mae_int = [mae_int[idx].mean().item() for idx in gene_groups.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gx = np.arange(len(group_names))
gw = 0.35

ax = axes[0]
ax.bar(gx - gw/2, group_pcc_prop, gw, label='Proportion', color='steelblue')
ax.bar(gx + gw/2, group_pcc_int, gw, label='Intensity', color='coral')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean PCC'); ax.set_title('PCC by Gene Group')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(gx - gw/2, group_mae_prop, gw, label='Proportion', color='steelblue')
ax.bar(gx + gw/2, group_mae_int, gw, label='Intensity', color='coral')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean MAE'); ax.set_title('MAE by Gene Group')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 5. GT vs Pred 직접 비교 시각화 =====

# --- 5-1. Random Sample Heatmap: GT vs Pred (Proportion & Intensity) ---
np.random.seed(42)
n_samples = 20
sample_idx = np.random.choice(len(all_gt_prop), n_samples, replace=False)
sample_idx = np.sort(sample_idx)

fig, axes = plt.subplots(2, 2, figsize=(22, 10))

# Proportion GT
im = axes[0, 0].imshow(all_gt_prop[sample_idx].numpy().T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
axes[0, 0].set_yticks(range(NUM_GENES)); axes[0, 0].set_yticklabels(MARKER_GENES, fontsize=8)
axes[0, 0].set_xlabel('Sample Index'); axes[0, 0].set_title('Proportion — Ground Truth')
plt.colorbar(im, ax=axes[0, 0], fraction=0.02)

# Proportion Pred
im = axes[0, 1].imshow(all_pred_prop[sample_idx].numpy().T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
axes[0, 1].set_yticks(range(NUM_GENES)); axes[0, 1].set_yticklabels(MARKER_GENES, fontsize=8)
axes[0, 1].set_xlabel('Sample Index'); axes[0, 1].set_title('Proportion — Prediction')
plt.colorbar(im, ax=axes[0, 1], fraction=0.02)

# Intensity GT
im = axes[1, 0].imshow(all_gt_int[sample_idx].numpy().T, aspect='auto', cmap='Oranges', vmin=0, vmax=1)
axes[1, 0].set_yticks(range(NUM_GENES)); axes[1, 0].set_yticklabels(MARKER_GENES, fontsize=8)
axes[1, 0].set_xlabel('Sample Index'); axes[1, 0].set_title('Intensity — Ground Truth')
plt.colorbar(im, ax=axes[1, 0], fraction=0.02)

# Intensity Pred
im = axes[1, 1].imshow(all_pred_int[sample_idx].numpy().T, aspect='auto', cmap='Oranges', vmin=0, vmax=1)
axes[1, 1].set_yticks(range(NUM_GENES)); axes[1, 1].set_yticklabels(MARKER_GENES, fontsize=8)
axes[1, 1].set_xlabel('Sample Index'); axes[1, 1].set_title('Intensity — Prediction')
plt.colorbar(im, ax=axes[1, 1], fraction=0.02)

plt.suptitle('Random 20 Samples: GT vs Prediction Heatmap', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- 5-2. Per-Gene Distribution Overlay (GT vs Pred) ---
fig, axes = plt.subplots(3, 5, figsize=(25, 12))
fig.suptitle('Proportion Distribution: GT (blue) vs Pred (red)', fontsize=14, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_prop[:, gi].numpy()
    pred_vals = all_pred_prop[:, gi].numpy()
    ax.hist(gt_vals, bins=50, alpha=0.5, color='steelblue', label='GT', density=True)
    ax.hist(pred_vals, bins=50, alpha=0.5, color='crimson', label='Pred', density=True)
    ax.set_title(f'{gene}', fontsize=10)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 5, figsize=(25, 12))
fig.suptitle('Intensity Distribution: GT (orange) vs Pred (red)', fontsize=14, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_int[:, gi].numpy()
    pred_vals = all_pred_int[:, gi].numpy()
    ax.hist(gt_vals, bins=50, alpha=0.5, color='orange', label='GT', density=True)
    ax.hist(pred_vals, bins=50, alpha=0.5, color='crimson', label='Pred', density=True)
    ax.set_title(f'{gene}', fontsize=10)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.show()

# --- 5-3. Random Sample Bar Chart: GT vs Pred per gene ---
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
fig.suptitle('Individual Sample Comparison: GT vs Pred', fontsize=14, y=1.02)
show_idx = np.random.choice(len(all_gt_prop), n_show, replace=False)

for col, si in enumerate(show_idx):
    x = np.arange(NUM_GENES)
    # Proportion
    ax = axes[0, col]
    ax.barh(x - 0.2, all_gt_prop[si].numpy(), 0.4, label='GT', color='steelblue', alpha=0.7)
    ax.barh(x + 0.2, all_pred_prop[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=7)
    ax.set_xlim(0, 1); ax.set_title(f'Sample {si}\n(Proportion)', fontsize=9)
    ax.legend(fontsize=6); ax.grid(True, alpha=0.3, axis='x')

    # Intensity
    ax = axes[1, col]
    ax.barh(x - 0.2, all_gt_int[si].numpy(), 0.4, label='GT', color='orange', alpha=0.7)
    ax.barh(x + 0.2, all_pred_int[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=7)
    ax.set_xlim(0, 1); ax.set_title(f'Sample {si}\n(Intensity)', fontsize=9)
    ax.legend(fontsize=6); ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# --- 5-4. Mean GT vs Mean Pred (전체 평균 비교) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(NUM_GENES)
w = 0.35

ax = axes[0]
ax.bar(x - w/2, all_gt_prop.mean(dim=0).numpy(), w, label='GT Mean', color='steelblue')
ax.bar(x + w/2, all_pred_prop.mean(dim=0).numpy(), w, label='Pred Mean', color='crimson')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('Mean Value'); ax.set_title('Proportion: GT Mean vs Pred Mean')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(x - w/2, all_gt_int.mean(dim=0).numpy(), w, label='GT Mean', color='orange')
ax.bar(x + w/2, all_pred_int.mean(dim=0).numpy(), w, label='Pred Mean', color='crimson')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('Mean Value'); ax.set_title('Intensity: GT Mean vs Pred Mean')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 6. Patch Image + GT vs Pred 비교 =====
np.random.seed(123)
n_show = 8
show_idx = np.random.choice(len(all_gt_prop), n_show, replace=False)

fig, axes = plt.subplots(n_show, 4, figsize=(28, 5 * n_show),
                         gridspec_kw={'width_ratios': [1, 1, 1.5, 1.5]})
fig.suptitle('Patch Image + GT vs Pred Comparison', fontsize=18, y=1.005)

for row, si in enumerate(show_idx):
    # --- 20x patch ---
    img_20x = np.array(Image.open(val_dataset.img_20x_paths[si]).convert('RGB'))
    ax = axes[row, 0]
    ax.imshow(img_20x)
    ax.set_title(f'Sample {si} — 20x Patch', fontsize=10)
    ax.axis('off')

    # --- 5x context patch ---
    img_5x = np.array(Image.open(val_dataset.img_5x_paths[si]).convert('RGB'))
    ax = axes[row, 1]
    ax.imshow(img_5x)
    ax.set_title(f'Sample {si} — 5x Context', fontsize=10)
    ax.axis('off')

    # --- Proportion: GT vs Pred ---
    x = np.arange(NUM_GENES)
    ax = axes[row, 2]
    ax.barh(x - 0.2, all_gt_prop[si].numpy(), 0.4, label='GT', color='steelblue', alpha=0.7)
    ax.barh(x + 0.2, all_pred_prop[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=8)
    ax.set_xlim(0, 1); ax.set_title('Proportion', fontsize=10)
    ax.legend(fontsize=7, loc='lower right'); ax.grid(True, alpha=0.3, axis='x')

    # --- Intensity: GT vs Pred ---
    ax = axes[row, 3]
    ax.barh(x - 0.2, all_gt_int[si].numpy(), 0.4, label='GT', color='orange', alpha=0.7)
    ax.barh(x + 0.2, all_pred_int[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=8)
    ax.set_xlim(0, 1); ax.set_title('Intensity', fontsize=10)
    ax.legend(fontsize=7, loc='lower right'); ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()